# Daily ESP fields and a mean three-dimensional eddy

This notebook reconstructs the ESP velocity field independently for every selected day of one eddy, recentres each field on its shallowest fitted centre, and then composites the evaluated velocity fields. It does not average ESP parameters before reconstruction.

The default onshore frame rotates each day using the local core-mean bathymetric gradient: positive axis 1 is onshore and positive axis 2 is alongshore. This prevents southward translation and slow changes in shelf orientation from smearing a persistent cross-shelf tilt. The mean fitted centre line is shown relative to the shallow centre. Because the authoritative TiltDir points from deep to shallow, an onshore TiltDir generally corresponds to deeper fitted centres lying on the negative/offshore side of the composite.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subfolder.')
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
import esp_composite_tools as ect

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300})
pd.set_option('display.max_columns', 100)

## Controls

Start with one long-lived eddy. Use DAY_RANGE to isolate the southward shelf-following part of its life. Set MAXIMUM_BATHYMETRY_M, for example to 2500 m, if you want to retain only days whose surface centre lies over the shelf or slope.

In [ ]:
EDDY_ID = 6
DAY_RANGE = (None, None)
MINIMUM_PROFILE_DEPTH_M = 500.0
MAXIMUM_BATHYMETRY_M = None

COMPOSITE_FRAME = 'onshore'  # 'onshore' or 'grid'
HORIZONTAL_UNITS = 'km'      # for 'Rc', also change HALF_WIDTH and GRID_STEP (e.g. 3 and 0.15)
HALF_WIDTH = 100.0
GRID_STEP = 5.0
MAXIMUM_DEPTH_M = 1000.0
MINIMUM_DEPTH_DAY_FRACTION = 0.70
DEPTH_LEVEL_INDICES = None   # e.g. (0, 4, 8, 12); None uses coverage threshold
ESP_ROOT = ect.DEFAULT_ESP_ROOT

## Load and filter one eddy

In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
surface_df, _ = tilt.load_tilt_tables(paths)
vertical_df = tilt.load_vert(paths, dic_form=False)

surface, profiles = ect.select_eddy_days(
    surface_df, vertical_df, EDDY_ID,
    day_range=DAY_RANGE,
    minimum_profile_depth_m=MINIMUM_PROFILE_DEPTH_M,
    maximum_bathymetry_m=MAXIMUM_BATHYMETRY_M,
    grid=grid,
)
print(
    f'{surface.Cyc.iloc[0]}{EDDY_ID}: {surface.Day.nunique()} selected days, '
    f'days {surface.Day.min():g}–{surface.Day.max():g}'
)

## Selected trajectory over bathymetry

Use this plot to refine DAY_RANGE before constructing the composite.

In [ ]:
pad = 60
xmin, xmax = surface.xc.min()-pad, surface.xc.max()+pad
ymin, ymax = surface.yc.min()-pad, surface.yc.max()+pad
inside = (
    (grid.X_grid >= xmin) & (grid.X_grid <= xmax)
    & (grid.Y_grid >= ymin) & (grid.Y_grid <= ymax)
)
bathy = np.where((grid.mask_rho == 1) & inside, grid.h/1e3, np.nan)
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
mesh = ax.contourf(grid.X_grid, grid.Y_grid, bathy, levels=25, cmap='Greys_r')
fig.colorbar(mesh, ax=ax, label='Depth (km)', shrink=.8)
points = ax.scatter(surface.xc, surface.yc, c=surface.Day, cmap='viridis', s=18)
ax.plot(surface.xc, surface.yc, color='0.3', lw=.7, alpha=.7)
fig.colorbar(points, ax=ax, label='Model day', shrink=.8)
ax.set(
    xlim=(xmin, xmax), ylim=(ymin, ymax), xlabel='x (km)', ylabel='y (km)',
    title=f'{surface.Cyc.iloc[0]}{EDDY_ID}: selected composite days', aspect='equal',
)
plt.show()

## Select exact fitted depths

No vertical interpolation is used between ESP profiles. By default, a fixed depth is retained when it occurs on at least the requested fraction of selected days. Alternatively, choose exact rows using DEPTH_LEVEL_INDICES.

In [ ]:
depth_catalogue = ect.depth_catalogue(profiles, MAXIMUM_DEPTH_M)
depths = ect.choose_depths(
    depth_catalogue,
    minimum_day_fraction=MINIMUM_DEPTH_DAY_FRACTION,
    depth_indices=DEPTH_LEVEL_INDICES,
)
depth_catalogue['selected'] = depth_catalogue.Depth.isin(depths)
display(depth_catalogue)
print('Selected exact depths:', ', '.join(f'{depth:g} m' for depth in depths))

## Reconstruct every daily field and composite them

Each daily ESP field is evaluated on the same relative grid. The mean and standard deviation are accumulated without retaining every full daily volume, which keeps memory use modest for long-lived eddies.

In [ ]:
composite = ect.build_eddy_composite(
    surface, profiles, grid, depths,
    frame=COMPOSITE_FRAME,
    half_width=HALF_WIDTH,
    grid_step=GRID_STEP,
    horizontal_units=HORIZONTAL_UNITS,
    esp_root=ESP_ROOT,
)
display(composite.daily)
display(pd.DataFrame({
    'Depth_m': composite.depths,
    'days_at_centre': composite.counts[:, len(composite.coordinate)//2, len(composite.coordinate)//2],
    'mean_axis1_offset': composite.centre_mean.axis1_offset,
    'mean_axis2_offset': composite.centre_mean.axis2_offset,
}))

## Is the daily tilt consistently onshore?

Zero means the authoritative deep-to-shallow TiltDir points onshore. The composite centre line uses shallow-to-deep offsets, so its sign is expected to be reversed: a negative deep cross-shelf offset means the shallow centre lies farther onshore.

In [ ]:
ect.plot_daily_alignment(composite)
plt.show()

angles = np.deg2rad(composite.daily.tilt_minus_axis1_deg.dropna())
mean_vector = np.mean(np.exp(1j*angles)) if len(angles) else np.nan
display(pd.Series({
    'days_with_direction': len(angles),
    'circular_mean_offset_deg': np.degrees(np.angle(mean_vector)) if len(angles) else np.nan,
    'resultant_length': np.abs(mean_vector) if len(angles) else np.nan,
    'fraction_within_45deg_onshore': np.mean(np.abs(np.degrees(np.angle(np.exp(1j*angles)))) <= 45) if len(angles) else np.nan,
}).to_frame('value'))

## Daily and mean vertical centre structure

Thin lines are individual days; the heavy line is the mean fitted centre at each exact depth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True, constrained_layout=True)
for _, part in composite.centres.groupby('Day'):
    axes[0].plot(part.axis1_offset, part.Depth, color='0.6', alpha=.2, lw=.7)
    axes[1].plot(part.axis2_offset, part.Depth, color='0.6', alpha=.2, lw=.7)
axes[0].plot(composite.centre_mean.axis1_offset, composite.depths, 'k.-', lw=2)
axes[1].plot(composite.centre_mean.axis2_offset, composite.depths, 'k.-', lw=2)
axis1 = 'Onshore' if COMPOSITE_FRAME == 'onshore' else 'Grid x'
axis2 = 'Alongshore' if COMPOSITE_FRAME == 'onshore' else 'Grid y'
axes[0].set(xlabel=f'{axis1} offset ({HORIZONTAL_UNITS})', ylabel='Depth (m)')
axes[1].set(xlabel=f'{axis2} offset ({HORIZONTAL_UNITS})')
for ax in axes:
    ax.axvline(0, color='0.3', lw=.7)
    ax.invert_yaxis()
plt.show()

## Composite velocity sections

In [ ]:
ect.plot_composite_sections(composite)
plt.show()

## Mean three-dimensional ESP velocity field

In [ ]:
ect.plot_composite_3d(composite, xy_step=3, z_step=2)
plt.show()

## Extension to multi-eddy composites

The same reconstruction can later be applied to pooled eddy-days selected by region, polarity, Rossby number, PV regime or shear. For that comparison, use radius-normalised horizontal units, preserve equal weighting at the eddy level rather than letting long-lived eddies dominate, and bootstrap entire eddies—not individual days—to retain within-eddy dependence.